## RQ-2: Einfluss von Prompt- und Programmiersprache

Dieses Notebook untersucht, ob die Wahl der Prompt-Sprache (Deutsch/Englisch) und der Programmiersprache (R/Python) einen messbaren Einfluss auf die Qualität der generierten Visualisierungen hat.

**Konfigurationen:** DE+R, DE+Python, EN+R, EN+Python

**Analysen:**
1. SEVQ-Total und SEVQ-Dimensionen pro Konfiguration – deskriptiv + Kruskal-Wallis-Test
2. Paarweise Mann-Whitney-U-Tests mit Bonferroni-Korrektur
3. A/B-Präferenzstudie: Welche Konfiguration bevorzugen menschliche Beurteiler?
4. Haupteffekte Sprache und Programmiersprache (zusammengefasst)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu, t as t_dist

---
### Konstanten und Konfiguration

Alle zentralen Konstanten werden hier einmalig definiert: SEVQ-Dimensionen, Modell-Mapping, Dateipfade und Konfigurationsbezeichnungen.

In [ ]:
METRICS = ['bugs', 'transformation', 'compliance', 'type', 'encoding', 'aesthetics']
MODEL_MAPPING = {
    'gpt-5': 'GPT-5',
    'gpt-4o': 'GPT-4o',
    'google/gemini-3-flash-preview': 'Gemini',
    'x-ai/grok-4.1-fast': 'Grok',
    'anthropic/claude-sonnet-4.5': 'Claude Sonnet'
}

BASE_PATH = Path("./src/resources/paper_results/50_dataset_run")
STUDY_PATH = Path("./results/rq_2/study_data.csv")
OUTPUT_DIR = Path("./results/rq_2/figures")
THESIS_FIG_DIR = OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIGS = {
    'DE_R': BASE_PATH / 'DE' / 'R' / 'output',
    'DE_Python': BASE_PATH / 'DE' / 'Python' / 'output',
    'EN_R': BASE_PATH / 'EN' / 'R' / 'output',
    'EN_Python': BASE_PATH / 'EN' / 'Python' / 'output',
}

CONFIG_LABELS = {
    'DE_R': 'DE & R',
    'DE_Python': 'DE & Py',
    'EN_R': 'EN & R',
    'EN_Python': 'EN & Py',
}

CONFIG_LABELS_LONG = {
    'DE_R': 'Deutsch & R',
    'DE_Python': 'Deutsch & Python',
    'EN_R': 'Englisch & R',
    'EN_Python': 'Englisch & Python',
}

Das A/B-Vergleichsdesign aus der Nutzerstudie: Jede Frage (13–24) zeigt den Teilnehmenden zwei Visualisierungen aus unterschiedlichen Konfigurationen. Die Zuordnung der Fragen zu Konfigurationspaaren ist wie folgt:

- Fragen 13–15: DE+R vs. DE+Python
- Fragen 16–18: DE+R vs. EN+R
- Fragen 19–21: DE+Python vs. EN+Python
- Fragen 22–24: EN+R vs. EN+Python

In [ ]:
AB_COMPARISONS = {
    'DE_R vs DE_Python': {
        'questions': [13, 14, 15],
        'variante_1': 'DE_R',
        'variante_2': 'DE_Python',
        'label': 'DE&R vs. DE&Py',
    },
    'DE_R vs EN_R': {
        'questions': [16, 17, 18],
        'variante_1': 'DE_R',
        'variante_2': 'EN_R',
        'label': 'DE&R vs. EN&R',
    },
    'DE_Python vs EN_Python': {
        'questions': [19, 20, 21],
        'variante_1': 'DE_Python',
        'variante_2': 'EN_Python',
        'label': 'DE&Py vs. EN&Py',
    },
    'EN_R vs EN_Python': {
        'questions': [22, 23, 24],
        'variante_1': 'EN_R',
        'variante_2': 'EN_Python',
        'label': 'EN&R vs. EN&Py',
    },
}

In [ ]:
# Farben für Konfigurationen
CONFIG_COLORS = {
    'DE_R': '#009B91',      # HTWG Teal (Accent 1)
    'DE_Python': '#334152',  # HTWG Dark Blue (Accent 3)
    'EN_R': '#546B86',       # HTWG Shaded Blue (Accent 4)
    'EN_Python': '#7990AB',  # HTWG Medium Blue (Accent 5)
}

---
### Hilfsfunktionen

Statistische Tests (Mann-Whitney-U, Kruskal-Wallis, Cohen's d, Binomial-KI) werden als Wrapper implementiert, um im weiteren Verlauf einheitlich aufgerufen werden zu können.

In [ ]:
def mann_whitney_u_test(x, y):
    """
    Mann-Whitney-U-Test (zweiseitig) via scipy.
    Gibt U-Statistik, z-Wert und p-Wert zurück.
    """
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    x, y = x[~np.isnan(x)], y[~np.isnan(y)]
    n1, n2 = len(x), len(y)
    if n1 == 0 or n2 == 0:
        return np.nan, np.nan, np.nan

    U_stat, p = mannwhitneyu(x, y, alternative='two-sided')
    # z-Approximation
    mu = n1 * n2 / 2
    sigma = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
    U = min(U_stat, n1 * n2 - U_stat)
    z = (U - mu) / sigma if sigma > 0 else 0.0
    return U, z, p


def kruskal_wallis_test(*groups):
    """
    Kruskal-Wallis H-Test via scipy.
    Gibt H-Statistik, Freiheitsgrade und p-Wert zurück.
    """
    groups = [np.asarray(g, dtype=float) for g in groups]
    groups = [g[~np.isnan(g)] for g in groups]
    groups = [g for g in groups if len(g) > 0]
    k = len(groups)
    if k < 2:
        return np.nan, np.nan, np.nan

    H, p = kruskal(*groups)
    df = k - 1
    return H, df, p


def cohens_d(x, y):
    """Effektstärke Cohen's d."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    x, y = x[~np.isnan(x)], y[~np.isnan(y)]
    n1, n2 = len(x), len(y)
    if n1 < 2 or n2 < 2:
        return np.nan
    pooled_std = np.sqrt(((n1 - 1) * np.var(x, ddof=1) + (n2 - 1) * np.var(y, ddof=1)) / (n1 + n2 - 2))
    if pooled_std == 0:
        return 0.0
    return (np.mean(x) - np.mean(y)) / pooled_std


def ci_95_mean(data):
    """95%-Konfidenzintervall des Mittelwerts (t-Verteilung via scipy)."""
    data = np.asarray(data, dtype=float)
    data = data[~np.isnan(data)]
    n = len(data)
    if n < 2:
        return np.nan, np.nan
    mean = np.mean(data)
    se = stats.sem(data)
    t_val = t_dist.ppf(0.975, df=n - 1)
    return mean - t_val * se, mean + t_val * se


def binomial_ci(k, n, z=1.96):
    """Wilson-Score-Intervall für Binomialanteil."""
    if n == 0:
        return 0.0, 0.0
    p_hat = k / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p_hat * (1 - p_hat) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

---
### Daten laden

Die SEVQ-Bewertungen werden aus den Experimentsordnern eingelesen. Zusätzlich werden die VER-Daten (Visualization Error Rate) pro Datensatz und die A/B-Präferenzdaten aus der Studie geladen.

In [ ]:
def load_sevq_data(output_path):
    """Lädt alle sevq.csv aus einem Konfigurationsordner."""
    dfs = []
    for folder in sorted(output_path.iterdir()):
        if not folder.is_dir():
            continue
        sevq_file = folder / 'sevq.csv'
        if sevq_file.exists():
            df = pd.read_csv(sevq_file)
            df['dataset'] = folder.name
            dfs.append(df)
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


def load_ver_data(output_path):
    """Lädt alle VER_0.csv (Iteration 0) aus einem Konfigurationsordner."""
    records = []
    for folder in sorted(output_path.iterdir()):
        if not folder.is_dir():
            continue
        ver_file = folder / 'VER_0.csv'
        if ver_file.exists():
            df = pd.read_csv(ver_file)
            n_vis = len(df)
            n_success = (df['ver_value'] == 1).sum()
            ver_rate = (n_success / n_vis * 100) if n_vis > 0 else np.nan
            records.append({
                'dataset': folder.name,
                'n_visualizations': n_vis,
                'n_success': int(n_success),
                'n_fail': int(n_vis - n_success),
                'ver_success_rate': ver_rate,
            })
    return pd.DataFrame(records)


def load_study_ab_data():
    """Lädt die A/B-Vergleichsantworten aus der Studiendatei."""
    df = pd.read_csv(STUDY_PATH, sep=';', encoding='utf-8')
    results = {}
    for comp_name, comp_info in AB_COMPARISONS.items():
        v1_count = 0
        v2_count = 0
        total = 0
        for q_num in comp_info['questions']:
            col_name = f'Frage {q_num} - Wählen Sie die Ihrer Meinung nach passendere Variante aus'
            if col_name not in df.columns:
                print(f"  WARNUNG: Spalte '{col_name}' nicht gefunden!")
                continue
            for val in df[col_name]:
                val_clean = str(val).replace('\u200b', '').strip()
                if 'Variante 1' in val_clean:
                    v1_count += 1
                    total += 1
                elif 'Variante 2' in val_clean:
                    v2_count += 1
                    total += 1
        results[comp_name] = {
            'variante_1': comp_info['variante_1'],
            'variante_2': comp_info['variante_2'],
            'v1_count': v1_count,
            'v2_count': v2_count,
            'total': total,
            'v1_pct': v1_count / total * 100 if total > 0 else 0,
            'v2_pct': v2_count / total * 100 if total > 0 else 0,
            'label': comp_info['label'],
        }
    return results

In [ ]:
sevq_data = {}
sevq_totals = {}  # config -> array of total scores (pro Visualisierung, gemittelt über Judges)
sevq_per_dim = {}  # config -> dict of dim -> array

for cfg, path in CONFIGS.items():
    df = load_sevq_data(path)
    sevq_data[cfg] = df
    if df.empty:
        sevq_totals[cfg] = np.array([])
        sevq_per_dim[cfg] = {m: np.array([]) for m in METRICS}
        continue

    # Total pro Visualisierung: Mittelwert über alle Judges pro vis
    df['total'] = df[METRICS].sum(axis=1)
    grouped = df.groupby(['dataset', 'vis_index'])['total'].mean()
    sevq_totals[cfg] = grouped.values

    sevq_per_dim[cfg] = {}
    for m in METRICS:
        grouped_m = df.groupby(['dataset', 'vis_index'])[m].mean()
        sevq_per_dim[cfg][m] = grouped_m.values

# --- 3b. VER-Daten laden ---
ver_data = {}
ver_rates = {}

for cfg, path in CONFIGS.items():
    df = load_ver_data(path)
    ver_data[cfg] = df
    ver_rates[cfg] = df['ver_success_rate'].values if not df.empty else np.array([])

# --- 3c. A/B-Daten laden ---
ab_results = load_study_ab_data()

---
### Deskriptive Statistik

Für jede der vier Konfigurationen werden Mittelwert, Standardabweichung, Median und 95%-Konfidenzintervall des SEVQ-Total-Scores ausgegeben.

In [ ]:
print(f"{'Konfiguration':<22} {'n':>5} {'M':>7} {'SD':>7} {'Mdn':>7} {'CI95 lo':>8} {'CI95 hi':>8}")
print("-" * 68)
for cfg in CONFIGS:
    data = sevq_totals[cfg]
    n = len(data)
    if n == 0:
        print(f"{CONFIG_LABELS_LONG[cfg]:<22} {0:>5}")
        continue
    m = np.mean(data)
    sd = np.std(data, ddof=1)
    mdn = np.median(data)
    ci_lo, ci_hi = ci_95_mean(data)
    print(f"{CONFIG_LABELS_LONG[cfg]:<22} {n:>5} {m:>7.2f} {sd:>7.2f} {mdn:>7.2f} {ci_lo:>8.2f} {ci_hi:>8.2f}")

print("\n--- SEVQ pro Dimension (Mittelwert ± SD über Judges) ---")
print(f"{'Konfiguration':<22}", end="")
for m in METRICS:
    print(f" {m:>16}", end="")
print()
print("-" * (22 + 17 * len(METRICS)))
for cfg in CONFIGS:
    print(f"{CONFIG_LABELS_LONG[cfg]:<22}", end="")
    for m in METRICS:
        data = sevq_per_dim[cfg][m]
        if len(data) == 0:
            print(f" {'—':>16}", end="")
        else:
            print(f" {np.mean(data):>6.2f}±{np.std(data, ddof=1):>4.2f}  ", end="")
    print()

print("\n--- VER Success Rate (Iteration 0) ---")
print(f"{'Konfiguration':<22} {'n_DS':>6} {'M':>7} {'SD':>7} {'Mdn':>7} {'CI95 lo':>8} {'CI95 hi':>8}")
print("-" * 72)
for cfg in CONFIGS:
    data = ver_rates[cfg]
    n = len(data)
    if n == 0:
        print(f"{CONFIG_LABELS_LONG[cfg]:<22} {0:>6}")
        continue
    m = np.mean(data)
    sd = np.std(data, ddof=1)
    mdn = np.median(data)
    ci_lo, ci_hi = ci_95_mean(data)
    print(f"{CONFIG_LABELS_LONG[cfg]:<22} {n:>6} {m:>7.2f} {sd:>7.2f} {mdn:>7.2f} {ci_lo:>8.2f} {ci_hi:>8.2f}")



---
### Statistische Tests

**Kruskal-Wallis-Test:** Gibt es überhaupt signifikante Unterschiede zwischen den vier Konfigurationen – sowohl beim SEVQ-Total als auch bei der VER?

In [ ]:
# Kruskal-Wallis über alle 4 Konfigurationen
groups_sevq = [sevq_totals[cfg] for cfg in CONFIGS if len(sevq_totals[cfg]) > 0]
H_sevq, df_sevq, p_sevq = kruskal_wallis_test(*groups_sevq)
print(f"\nKruskal-Wallis (SEVQ-Total): H = {H_sevq:.4f}, df = {df_sevq}, p = {p_sevq:.4f}")

groups_ver = [ver_rates[cfg] for cfg in CONFIGS if len(ver_rates[cfg]) > 0]
H_ver, df_ver, p_ver = kruskal_wallis_test(*groups_ver)
print(f"Kruskal-Wallis (VER):        H = {H_ver:.4f}, df = {df_ver}, p = {p_ver:.4f}")

**Paarweise Mann-Whitney-U-Tests (SEVQ-Total):** Welche Konfigurationspaare unterscheiden sich signifikant? Die p-Werte werden mit Bonferroni korrigiert (6 Vergleiche).

In [ ]:
print("\n--- Paarweise Mann-Whitney-U-Tests (SEVQ-Total) ---")
print(f"{'Vergleich':<30} {'U':>10} {'z':>8} {'p':>10} {'p_adj':>10} {'d':>7} {'sig':>5}")
print("-" * 85)
cfg_keys = list(CONFIGS.keys())
n_comparisons = 6  # 4 choose 2

pairwise_results_sevq = []
for i in range(len(cfg_keys)):
    for j in range(i + 1, len(cfg_keys)):
        c1, c2 = cfg_keys[i], cfg_keys[j]
        if len(sevq_totals[c1]) == 0 or len(sevq_totals[c2]) == 0:
            continue
        U, z, p = mann_whitney_u_test(sevq_totals[c1], sevq_totals[c2])
        p_adj = min(1.0, p * n_comparisons)
        d = cohens_d(sevq_totals[c1], sevq_totals[c2])
        sig = "*" if p_adj < 0.05 else ("(*)" if p_adj < 0.10 else "")
        label = f"{CONFIG_LABELS[c1]} vs. {CONFIG_LABELS[c2]}"
        print(f"{label:<30} {U:>10.1f} {z:>8.3f} {p:>10.4f} {p_adj:>10.4f} {d:>7.3f} {sig:>5}")
        pairwise_results_sevq.append({
            'comp': label, 'c1': c1, 'c2': c2,
            'U': U, 'z': z, 'p': p, 'p_adj': p_adj, 'd': d
        })

**Paarweise Mann-Whitney-U-Tests (VER):** Dieselbe Analyse für die Visualization Error Rate.

In [ ]:
print("\n--- Paarweise Mann-Whitney-U-Tests (VER) ---")
print(f"{'Vergleich':<30} {'U':>10} {'z':>8} {'p':>10} {'p_adj':>10} {'d':>7} {'sig':>5}")
print("-" * 85)

pairwise_results_ver = []
for i in range(len(cfg_keys)):
    for j in range(i + 1, len(cfg_keys)):
        c1, c2 = cfg_keys[i], cfg_keys[j]
        if len(ver_rates[c1]) == 0 or len(ver_rates[c2]) == 0:
            continue
        U, z, p = mann_whitney_u_test(ver_rates[c1], ver_rates[c2])
        p_adj = min(1.0, p * n_comparisons)
        d = cohens_d(ver_rates[c1], ver_rates[c2])
        sig = "*" if p_adj < 0.05 else ("(*)" if p_adj < 0.10 else "")
        label = f"{CONFIG_LABELS[c1]} vs. {CONFIG_LABELS[c2]}"
        print(f"{label:<30} {U:>10.1f} {z:>8.3f} {p:>10.4f} {p_adj:>10.4f} {d:>7.3f} {sig:>5}")
        pairwise_results_ver.append({
            'comp': label, 'c1': c1, 'c2': c2,
            'U': U, 'z': z, 'p': p, 'p_adj': p_adj, 'd': d
        })

**A/B-Studie:** Wie oft haben Studienteilnehmer die jeweilige Variante bevorzugt? Ausgegeben werden absolute Häufigkeiten und Prozentwerte mit 95%-Binomial-KI.

In [ ]:
print(f"\n{'Vergleich':<25} {'Var.1':<10} {'n₁':>4} {'Var.2':<10} {'n₂':>4} {'Total':>6} {'%1':>6} {'%2':>6} {'CI95(%)':>14}")
print("-" * 95)
for comp_name, res in ab_results.items():
    ci_lo, ci_hi = binomial_ci(res['v2_count'], res['total'])
    v1_label = CONFIG_LABELS[res['variante_1']]
    v2_label = CONFIG_LABELS[res['variante_2']]
    print(f"{res['label']:<25} {v1_label:<10} {res['v1_count']:>4} {v2_label:<10} {res['v2_count']:>4} "
          f"{res['total']:>6} {res['v1_pct']:>5.1f}% {res['v2_pct']:>5.1f}% [{ci_lo*100:>5.1f},{ci_hi*100:>5.1f}]")


---
### Haupteffekte: Sprache und Programmiersprache

Um von Konfigurationsebene auf Faktorebene zu abstrahieren, werden die Daten nach Prompt-Sprache (DE vs. EN) und Programmiersprache (R vs. Python) zusammengefasst und verglichen.

In [ ]:
# SEVQ
de_sevq = np.concatenate([sevq_totals['DE_R'], sevq_totals['DE_Python']])
en_sevq = np.concatenate([sevq_totals['EN_R'], sevq_totals['EN_Python']])
r_sevq = np.concatenate([sevq_totals['DE_R'], sevq_totals['EN_R']])
py_sevq = np.concatenate([sevq_totals['DE_Python'], sevq_totals['EN_Python']])

print(f"\nSEVQ-Total nach Promptsprache:")
print(f"  Deutsch:  M={np.mean(de_sevq):.2f}, SD={np.std(de_sevq, ddof=1):.2f}, n={len(de_sevq)}")
print(f"  Englisch: M={np.mean(en_sevq):.2f}, SD={np.std(en_sevq, ddof=1):.2f}, n={len(en_sevq)}")
U, z, p = mann_whitney_u_test(de_sevq, en_sevq)
d = cohens_d(de_sevq, en_sevq)
print(f"  Mann-Whitney U={U:.1f}, z={z:.3f}, p={p:.4f}, d={d:.3f}")

print(f"\nSEVQ-Total nach Programmiersprache:")
print(f"  R:      M={np.mean(r_sevq):.2f}, SD={np.std(r_sevq, ddof=1):.2f}, n={len(r_sevq)}")
print(f"  Python: M={np.mean(py_sevq):.2f}, SD={np.std(py_sevq, ddof=1):.2f}, n={len(py_sevq)}")
U, z, p = mann_whitney_u_test(r_sevq, py_sevq)
d = cohens_d(r_sevq, py_sevq)
print(f"  Mann-Whitney U={U:.1f}, z={z:.3f}, p={p:.4f}, d={d:.3f}")

# VER
de_ver = np.concatenate([ver_rates['DE_R'], ver_rates['DE_Python']])
en_ver = np.concatenate([ver_rates['EN_R'], ver_rates['EN_Python']])
r_ver = np.concatenate([ver_rates['DE_R'], ver_rates['EN_R']])
py_ver = np.concatenate([ver_rates['DE_Python'], ver_rates['EN_Python']])

print(f"\nVER nach Promptsprache:")
print(f"  Deutsch:  M={np.mean(de_ver):.2f}%, SD={np.std(de_ver, ddof=1):.2f}")
print(f"  Englisch: M={np.mean(en_ver):.2f}%, SD={np.std(en_ver, ddof=1):.2f}")
U, z, p = mann_whitney_u_test(de_ver, en_ver)
d = cohens_d(de_ver, en_ver)
print(f"  Mann-Whitney U={U:.1f}, z={z:.3f}, p={p:.4f}, d={d:.3f}")

print(f"\nVER nach Programmiersprache:")
print(f"  R:      M={np.mean(r_ver):.2f}%, SD={np.std(r_ver, ddof=1):.2f}")
print(f"  Python: M={np.mean(py_ver):.2f}%, SD={np.std(py_ver, ddof=1):.2f}")
U, z, p = mann_whitney_u_test(r_ver, py_ver)
d = cohens_d(r_ver, py_ver)
print(f"  Mann-Whitney U={U:.1f}, z={z:.3f}, p={p:.4f}, d={d:.3f}")

---
### Abbildungen für die Thesis

Die folgenden Plots werden als SVG gespeichert und direkt in die Thesis eingebunden.

**Abbildung: SEVQ-Total-Score pro Konfiguration** – Boxplot mit allen Einzelwerten.

In [ ]:
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.dpi': 150,
})

fig, ax = plt.subplots(figsize=(8, 5))
bp_data = [sevq_totals[cfg] for cfg in CONFIGS]
bp_labels = [CONFIG_LABELS[cfg] for cfg in CONFIGS]
bp_colors = [CONFIG_COLORS[cfg] for cfg in CONFIGS]

bp = ax.boxplot(bp_data, labels=bp_labels, patch_artist=True, widths=0.5,
                medianprops=dict(color='black', linewidth=1.5),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2))
for patch, color in zip(bp['boxes'], bp_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Mittelwert-Marker
for i, cfg in enumerate(CONFIGS):
    mean_val = np.mean(sevq_totals[cfg]) if len(sevq_totals[cfg]) > 0 else 0
    ax.plot(i + 1, mean_val, 'D', color='white', markeredgecolor='black',
            markersize=7, zorder=5)

ax.set_ylabel('SEVQ-Total (Ø über Judges)')
ax.set_xlabel('Konfiguration')
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

# n-Labels
for i, cfg in enumerate(CONFIGS):
    n = len(sevq_totals[cfg])
    ax.text(i + 1, ax.get_ylim()[0] - 1.5, f'n={n}', ha='center', va='top', fontsize=9, color='gray')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'rq2-sevq-total-boxplot.svg', format='svg', bbox_inches='tight')
fig.savefig(THESIS_FIG_DIR / '05-rq2-sevq-total-boxplot.svg', format='svg', bbox_inches='tight')
plt.close(fig)
print("\n[OK] rq2-sevq-total-boxplot.svg")


**Abbildung: SEVQ-Dimensionen pro Konfiguration** – Heatmap der mittleren Bewertungen pro Dimension und Konfiguration.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
heatmap_data = np.zeros((len(CONFIGS), len(METRICS)))
for i, cfg in enumerate(CONFIGS):
    for j, m in enumerate(METRICS):
        data = sevq_per_dim[cfg][m]
        heatmap_data[i, j] = np.mean(data) if len(data) > 0 else 0

sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn',
            xticklabels=[m.capitalize() for m in METRICS],
            yticklabels=[CONFIG_LABELS[cfg] for cfg in CONFIGS],
            vmin=5, vmax=10, linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Ø SEVQ-Score'},
            ax=ax)
ax.set_xlabel('SEVQ-Dimension')
ax.set_ylabel('Konfiguration')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'rq2-sevq-dimensions-heatmap.svg', format='svg', bbox_inches='tight')
fig.savefig(THESIS_FIG_DIR / '05-rq2-sevq-dimensions-heatmap.svg', format='svg', bbox_inches='tight')
plt.close(fig)
print("[OK] rq2-sevq-dimensions-heatmap.svg")

**Abbildung: Visualization Error Rate (VER) pro Konfiguration** – Boxplot analog zum SEVQ-Plot.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bp_data_ver = [ver_rates[cfg] for cfg in CONFIGS]
bp = ax.boxplot(bp_data_ver, labels=bp_labels, patch_artist=True, widths=0.5,
                medianprops=dict(color='black', linewidth=1.5),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2))
for patch, color in zip(bp['boxes'], bp_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for i, cfg in enumerate(CONFIGS):
    mean_val = np.mean(ver_rates[cfg]) if len(ver_rates[cfg]) > 0 else 0
    ax.plot(i + 1, mean_val, 'D', color='white', markeredgecolor='black',
            markersize=7, zorder=5)

ax.set_ylabel('VER Success Rate (%)')
ax.set_xlabel('Konfiguration')
ax.set_ylim(-5, 105)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

for i, cfg in enumerate(CONFIGS):
    n = len(ver_rates[cfg])
    ax.text(i + 1, ax.get_ylim()[0] - 7, f'n={n}', ha='center', va='top', fontsize=9, color='gray')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'rq2-ver-boxplot.svg', format='svg', bbox_inches='tight')
fig.savefig(THESIS_FIG_DIR / '05-rq2-ver-boxplot.svg', format='svg', bbox_inches='tight')
plt.close(fig)
print("[OK] rq2-ver-boxplot.svg")

**Abbildung: A/B-Präferenzen** – Horizontales Balkendiagramm zeigt, wie oft Variante 1 bzw. Variante 2 bevorzugt wurde.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
comp_labels = []
v1_pcts = []
v2_pcts = []
v1_labels_list = []
v2_labels_list = []

for comp_name, res in ab_results.items():
    comp_labels.append(res['label'])
    v1_pcts.append(res['v1_pct'])
    v2_pcts.append(res['v2_pct'])
    v1_labels_list.append(CONFIG_LABELS[res['variante_1']])
    v2_labels_list.append(CONFIG_LABELS[res['variante_2']])

y_pos = np.arange(len(comp_labels))
bar_height = 0.35

# Zeichne als divergierende Balken von der Mitte (50%)
v1_bars = ax.barh(y_pos, [-p for p in v1_pcts], bar_height, left=0,
                  color='#009B91', alpha=0.8, label='Variante 1')
v2_bars = ax.barh(y_pos, v2_pcts, bar_height, left=0,
                  color='#334152', alpha=0.8, label='Variante 2')

# Labels auf den Balken
for i, (v1p, v2p) in enumerate(zip(v1_pcts, v2_pcts)):
    ax.text(-v1p / 2, i, f'{v1p:.0f}%\n({v1_labels_list[i]})', ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')
    ax.text(v2p / 2, i, f'{v2p:.0f}%\n({v2_labels_list[i]})', ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')

ax.set_yticks(y_pos)
ax.set_yticklabels(comp_labels)
ax.set_xlabel('Präferenz (%)')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlim(-100, 100)
ax.set_xticks([-100, -75, -50, -25, 0, 25, 50, 75, 100])
ax.set_xticklabels(['100', '75', '50', '25', '0', '25', '50', '75', '100'])
ax.grid(axis='x', alpha=0.3)
ax.set_axisbelow(True)

# 50%-Linie
ax.axvline(x=-50, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(x=50, color='gray', linewidth=0.5, linestyle='--')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'rq2-ab-preferences.svg', format='svg', bbox_inches='tight')
fig.savefig(THESIS_FIG_DIR / '05-rq2-ab-preferences.svg', format='svg', bbox_inches='tight')
plt.close(fig)
print("[OK] rq2-ab-preferences.svg")

**Abbildung: Zusammenfassender Vergleich** – Drei Panels nebeneinander: SEVQ-Mittelwerte mit KI, VER-Mittelwerte und A/B-Präferenzen im Überblick.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Panel 1: SEVQ-Total Mittelwerte mit CI
ax = axes[0]
means = []
cis = []
for cfg in CONFIGS:
    data = sevq_totals[cfg]
    means.append(np.mean(data) if len(data) > 0 else 0)
    lo, hi = ci_95_mean(data)
    cis.append((lo, hi))

x = np.arange(len(CONFIGS))
colors = [CONFIG_COLORS[cfg] for cfg in CONFIGS]
bars = ax.bar(x, means, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
for i, (lo, hi) in enumerate(cis):
    if not np.isnan(lo):
        ax.errorbar(i, means[i], yerr=[[means[i] - lo], [hi - means[i]]],
                     color='black', capsize=5, capthick=1.5, linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels([CONFIG_LABELS[cfg] for cfg in CONFIGS], fontsize=10)
ax.set_ylabel('SEVQ-Total (Ø)')
ax.set_title('(a) SEVQ-Total')
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

# Panel 2: VER Mittelwerte mit CI
ax = axes[1]
means_ver = []
cis_ver = []
for cfg in CONFIGS:
    data = ver_rates[cfg]
    means_ver.append(np.mean(data) if len(data) > 0 else 0)
    lo, hi = ci_95_mean(data)
    cis_ver.append((lo, hi))

bars = ax.bar(x, means_ver, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
for i, (lo, hi) in enumerate(cis_ver):
    if not np.isnan(lo):
        ax.errorbar(i, means_ver[i], yerr=[[means_ver[i] - lo], [hi - means_ver[i]]],
                     color='black', capsize=5, capthick=1.5, linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels([CONFIG_LABELS[cfg] for cfg in CONFIGS], fontsize=10)
ax.set_ylabel('VER Success Rate (%)')
ax.set_title('(b) VER Success Rate')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

# Panel 3: A/B-Präferenz (gesammelte Wins)
ax = axes[2]
# Zähle Gesamtpräferenz pro Config
config_wins = {cfg: 0 for cfg in CONFIGS}
config_total = {cfg: 0 for cfg in CONFIGS}
for comp_name, res in ab_results.items():
    config_wins[res['variante_1']] += res['v1_count']
    config_wins[res['variante_2']] += res['v2_count']
    config_total[res['variante_1']] += res['total']
    config_total[res['variante_2']] += res['total']

win_rates = []
win_cis = []
for cfg in CONFIGS:
    if config_total[cfg] > 0:
        wr = config_wins[cfg] / config_total[cfg] * 100
        lo, hi = binomial_ci(config_wins[cfg], config_total[cfg])
        win_rates.append(wr)
        win_cis.append((lo * 100, hi * 100))
    else:
        win_rates.append(0)
        win_cis.append((0, 0))

bars = ax.bar(x, win_rates, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
for i, (lo, hi) in enumerate(win_cis):
    ax.errorbar(i, win_rates[i], yerr=[[win_rates[i] - lo], [hi - win_rates[i]]],
                color='black', capsize=5, capthick=1.5, linewidth=1.5)
ax.axhline(y=50, color='gray', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels([CONFIG_LABELS[cfg] for cfg in CONFIGS], fontsize=10)
ax.set_ylabel('Präferenzrate (%)')
ax.set_title('(c) Teilnehmer-Präferenz')
ax.set_ylim(0, 80)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'rq2-summary-comparison.svg', format='svg', bbox_inches='tight')
fig.savefig(THESIS_FIG_DIR / '05-rq2-summary-comparison.svg', format='svg', bbox_inches='tight')
plt.close(fig)
print("[OK] rq2-summary-comparison.svg")